## Notebook to try out my idea to create embeddings for my dashboard for task 6 using plotly

In [0]:
%python
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Läsa in data ifrån mina två growth marts i transformations/gold
df_official = spark.sql(
    """
    SELECT 
        SUM(total_events_hosted) AS total_events,
        SUM(total_global_finishers) AS total_finishers
    FROM marathos.gold.mart_event_growth_official"""
).toPandas()

df_verified = spark.sql(
    """
    SELECT 
        SUM(total_global_finishers) AS verified_finishers
    FROM marathos.gold.mart_event_growth_verified"""
).toPandas()

- Why use `.toPandas()` in the cell above?
    - Reason is because im aggregating it down to ONE row which is trivial for pandas. It WOULD start to become a problem if I would try to aggregate 6.8mil rows with pandas.

In [0]:
total_events = int(df_official["total_events"].iloc[0])
total_finishers = int(df_official["total_finishers"].iloc[0])

verified = int(df_verified["verified_finishers"].iloc[0])
no_show_rate = round((total_finishers - verified) / total_finishers * 100, 1)

print(f"Events: {total_events:,}") # :, för att separera siffrorna för läsbarhet
print(f"Official: {total_finishers:,}")
print(f"Verified: {verified:,}")
print(f"No shows: {no_show_rate}%")

- Using `:,` in an f-string is a format specifier. Python automatically formats big numbers with thousand separators.

In [0]:
%python
# Byggandet av mina KPI kort
fig = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}] * 4])

fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_events,
        title={"text": "Total Events<br><span style='font-size:0.8em'>All time</span>"},
        number={"valueformat": ","},
    ),
    row=1,
    col=1,
)

